# Structured Output

In [ ]:
from langchain_ollama import ChatOllama

model_name = "qwen3-0.6b:latest"
model = ChatOllama(model=model_name, temperature=0)

## Pydantic

In [2]:
from pydantic import BaseModel, Field

class Movie(BaseModel):
    title:str = Field(description="The title of the movie")
    year:int = Field(description="This year the movie was released")
    director:str = Field(description="The director of the movie")
    rating:float = Field(description="The movies rating out of 10")

In [3]:
model_with_structure = model.with_structured_output(Movie)

model_with_structure

_ChatModelBinding(bound=ChatOllama(metadata={'lc_versions': {'langchain-core': '1.6.2', 'langchain': '1.3.15'}}, model='qwen3-0.6b:latest', temperature=0.0), kwargs={'format': {'properties': {'title': {'description': 'The title of the movie', 'title': 'Title', 'type': 'string'}, 'year': {'description': 'This year the movie was released', 'title': 'Year', 'type': 'integer'}, 'director': {'description': 'The director of the movie', 'title': 'Director', 'type': 'string'}, 'rating': {'description': 'The movies rating out of 10', 'title': 'Rating', 'type': 'number'}}, 'required': ['title', 'year', 'director', 'rating'], 'title': 'Movie', 'type': 'object'}, 'ls_structured_output_format': {'kwargs': {'method': 'json_schema'}, 'schema': <class '__main__.Movie'>}}, config={}, config_factories=[])
| PydanticOutputParser(pydantic_object=<class '__main__.Movie'>)

In [4]:
model_with_structure.invoke("Tell me about the movie 'Inception'")

Movie(title='Inception', year=2010, director='Christopher Nolan', rating=8.5)

In [5]:
model.invoke("Tell me about the movie 'Inception'")

AIMessage(content='**Inception** (2010) is a science fiction action film directed by Christopher Nolan. It follows a thief named Christopher Nolan, who enters the dreams of others to steal secrets. The film explores themes of reality, dreams, and the subconscious, with a complex narrative involving a protagonist and a partner. The movie is known for its stunning visual effects and intricate plot, making it a landmark in modern cinema.', additional_kwargs={}, response_metadata={'model': 'qwen3-0.6b:latest', 'created_at': '2026-09-13T07:06:06.5934566Z', 'done': True, 'done_reason': 'stop', 'total_duration': 33294131800, 'load_duration': 755900600, 'prompt_eval_count': 17, 'prompt_eval_duration': 63008000, 'eval_count': 334, 'eval_duration': 32420489000, 'logprobs': None, 'model_name': 'qwen3-0.6b:latest', 'model_provider': 'ollama'}, id='lc_run--01a09995-e724-76e3-9179-eb511caaa9b0-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 17, 'output_tokens': 334, 'total_

In [6]:
from pydantic import BaseModel, Field

class Movie(BaseModel):
    title:str = Field(..., description="The title of the movie")
    year:int = Field(..., description="This year the movie was released")
    director:str = Field(..., description="The director of the movie")
    rating:float = Field(..., description="The movies rating out of 10")

model_with_structure = model.with_structured_output(Movie, include_raw=True)
response = model_with_structure.invoke("Tell me about the movie 'Inception'")

print(response)

{'raw': AIMessage(content='{\n  "director": "Christopher Nolan",\n  "rating": 8.5,\n  "title": "Inception",\n  "year": 2010\n}', additional_kwargs={}, response_metadata={'model': 'qwen3-0.6b:latest', 'created_at': '2026-09-13T07:06:27.6039027Z', 'done': True, 'done_reason': 'stop', 'total_duration': 20910039700, 'load_duration': 411380100, 'prompt_eval_count': 17, 'prompt_eval_duration': 113535000, 'eval_count': 295, 'eval_duration': 20313346000, 'logprobs': None, 'model_name': 'qwen3-0.6b:latest', 'model_provider': 'ollama'}, id='lc_run--01a09996-69cd-7e50-ab66-2e78358faf9c-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 17, 'output_tokens': 295, 'total_tokens': 312}), 'parsed': Movie(title='Inception', year=2010, director='Christopher Nolan', rating=8.5), 'parsing_error': None}


### Nested Structure

In [7]:
from pydantic import BaseModel, Field

class Actor(BaseModel):
    name: str
    role: str

class MovieDetails(BaseModel):
    title:str
    year:int
    cast:list[Actor]
    genres:list[str]
    budget:float|None = Field(None, description="Budget in millions USD")

model_with_structure = model.with_structured_output(MovieDetails)
response = model_with_structure.invoke("Tell me about the movie 'Inception'")

print(response)

title='Inception' year=2010 cast=[Actor(name='Christopher Nolan', role='Director'), Actor(name='Jamie Foxx', role='The thief'), Actor(name='Evan Parks', role='The partner'), Actor(name='Jude Law', role='The dreamer')] genres=['Sci-Fi', 'Action'] budget=inf


## TypeDict

In [8]:
from typing_extensions import TypedDict, Annotated

class MovieDict(TypedDict):
    """A movie with details."""
    title: Annotated[str, ..., "The title of the movie"]
    year: Annotated[int, ..., "This year the movie was released"]
    director: Annotated[str, ..., "The director of the movie"]
    rating: Annotated[float, ..., "The movies rating out of 10"]

In [9]:
model_with_structure = model.with_structured_output(MovieDict)
response = model_with_structure.invoke("Tell me about the movie 'Inception'")

print(response)

{'director': 'Christopher Nolan', 'rating': 8.5, 'title': 'Inception', 'year': 2010}


### Nested Structure

In [10]:
class Actor(TypedDict):
    name: str
    role: str

class MovieDetails(TypedDict):
    title:str
    year:int
    cast:list[Actor]
    genres:list[str]
    budget:float|None = Field(None, description="Budget in millions USD")

model_with_typeddict = model.with_structured_output(MovieDetails)
response = model_with_typeddict.invoke("Tell me about the movie 'Avengers'")

response

{'budget': 1.55,
 'cast': [{'name': 'Elena Farris', 'role': 'Iron Man'},
  {'name': 'Tony Stark', 'role': 'Iron Man'},
  {'name': 'Carol Kyle', 'role': 'Iron Man'}],
 'genres': ['Action', 'Sci-Fi'],
 'title': 'Avengers',
 'year': 2012}

## Data Classes

In [11]:
from langchain_ollama import ChatOllama

model = ChatOllama(model=model_name, temperature=0)

In [12]:
from pydantic import Field, BaseModel
from langchain.agents import create_agent

class ContactInfo(BaseModel):
    """Contact information for a person."""
    name: str = Field(description="The name of the person")
    email: str = Field(description="The email address of the person")
    phone: str = Field(description="The phone number of the person")

agent = create_agent(model=model, response_format=ContactInfo)
messages = [{"role": "user", "content": "Extract contact info from: Mohammad Beiro, beiro@example.com, (888) 123-4567"}]

result = agent.invoke({"messages": messages})

print(result["structured_response"])

name='Mohammad Beiro' email='beiro@example.com' phone='(888) 123-4567'


In [13]:
from typing_extensions import TypedDict, Annotated
from langchain.agents import create_agent

class ContactInfo(TypedDict):
    """Contact information for a person."""
    name: Annotated[str, ..., "The name of the person"]
    email: Annotated[str, ..., "The email address of the person"]
    phone: Annotated[str, ..., "The phone number of the person"]

agent = create_agent(model=model, response_format=ContactInfo)
messages = [{"role": "user", "content": "Extract contact info from: Mohammad Beiro, beiro@example.com, (888) 123-4567"}]

result = agent.invoke({"messages": messages})

print(result["structured_response"])

{'name': 'Mohammad Beiro', 'email': 'beiro@example.com', 'phone': '(888) 123-4567'}


In [14]:
from dataclasses import dataclass
from langchain.agents import create_agent

@dataclass
class ContactInfo:
    """Contact information for a person."""
    name: str
    email: str
    phone: str

agent = create_agent(model=model, response_format=ContactInfo)
messages = [{"role": "user", "content": "Extract contact info from: Mohammad Beiro, beiro@example.com, (888) 123-4567"}]

result = agent.invoke({"messages": messages})

print(result["structured_response"])

ContactInfo(name='Mohammad Beiro', email='beiro@example.com', phone='(888) 123-4567')
